In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import lbpscreen as lbp
from pathlib import Path

models = lbp.load_models("../data/processed/curated_models")
m = models["E_coli_Nissle1917"]
hp = lbp.load_diet("../data/raw/diets/high-protein.csv")
sol = m.optimize()
print(sol.objective_value)
print(sol.status)

93.91435011269724
optimal


In [2]:
print("EX_o2_e_" in m.reactions)
print(m.reactions.EX_o2_e_.lower_bound)

True
-1000.0


In [3]:
with m:
    m.reactions.EX_o2_e_.lower_bound = 0
    print(m.reactions.EX_o2_e_.lower_bound)
    print("Growth at anaerobic conditions:", m.slim_optimize())
print(m.reactions.EX_o2_e_.lower_bound)
print("Growth at aerobic conditions:", m.slim_optimize())

0
Growth at anaerobic conditions: 93.91435011269724
-1000.0
Growth at aerobic conditions: 93.91435011269724


In [4]:
lbp.anaerobic(m)
print(m.reactions.EX_o2_e_.lower_bound)
print(m.slim_optimize())

AttributeError: module 'lbpscreen' has no attribute 'anaerobic'

In [ ]:
small = {"EX_glc_D_e_": 10.0, "EX_fe2_e_": 2.0}
scaled = {k: v * 0.1 for k, v in small.items()}
print(scaled)

In [ ]:
with m:
    missing = lbp.apply_medium(m, small, False)
    print("missing from model:", missing)
    print("growth:", m.slim_optimize())

In [ ]:
with m:
    missing = lbp.apply_diet(m, hp, scale=1.0)
    print(len(missing), "diet items not in Nissle; growth:", m.slim_optimize())

In [ ]:
m = models["E_coli_Nissle1917"]
print("Growth without the diet:", m.slim_optimize())

with m:
    missing = lbp.apply_diet(m, hp)            # apply the high-protein diet
    base = m.slim_optimize()
    print("Growth on diet:", base)

    rescuers = []
    for r in m.exchanges:
        if r.lower_bound == 0:                        # currently closed for uptake
            with m:
                r.lower_bound = -10                   # try reopening just this one
                if m.slim_optimize() > 1e-6 and base < 1e-6:
                    rescuers.append(r.id)
    print(f"{len(rescuers)} closed exchanges each restore growth:")
    print(rescuers)

In [ ]:
with m:
    lbp.apply_diet(m, hp)
    for e in ["EX_so4_e_", "EX_cobalt2_e_"]:
        m.reactions.get_by_id(e).lower_bound = -1.0
    print("growth:", m.slim_optimize())     # ~12.5

In [ ]:
with m:
    lbp.apply_diet(m, hp)
    for e in ["EX_so4_e_", "EX_cobalt2_e_"]:
        m.reactions.get_by_id(e).lower_bound = -10.0
    base = m.slim_optimize()
    with m:
        m.reactions.EX_glc_D_e_.lower_bound = 0
        print("full:", round(base, 3), " no glucose:", round(m.slim_optimize(), 3))

In [ ]:
rep = lbp.diet_report(m, hp)
print("Growth on diet:", rep.attrs["growth_on_diet"])
print(rep["In Model"].value_counts())
lbp.essential_diet_components(m, hp)

In [ ]:
lbp.growth_on_diets(models, {"high_protein": hp})